# Notebook 2 — Training SSD MobileNet V2 FPNLite 640×640
## Smart Sniper Spotter · Target Detection Module

This notebook trains the primary detection model:
- **Architecture:** SSD MobileNet V2 with FPN-Lite head
- **Input resolution:** 640×640
- **Target hardware:** Coral EdgeTPU (int8 quantization)
- **Single class:** person

### Kaggle Setup
1. Create a new Kaggle notebook — set **Accelerator** to **GPU T4 ×2** (or GPU P100)
2. **Add Input → Notebooks tab** → search for your "Data Preparation" notebook and attach it
3. **Enable Internet** in settings (needed to download pretrained checkpoint)
4. Upload this notebook and run all cells

### Training Strategy
- Fine-tune from a COCO-pretrained checkpoint (transfer learning)
- Apply augmentations targeting our operational conditions
- Train with quantization-aware parameters for EdgeTPU export
- Save checkpoints every 1000 steps for session resume capability


## 1. Environment Setup

In [1]:
# Cell #1: Environment Setup for TF2 Object Detection API on Kaggle
#
# Kaggle image: Python 3.12, TF 2.19.x (Keras 3), numpy 2.0.2, JAX pre-installed,
#               system protoc == 3.12 (too old for modern protobuf runtime).
#
# Known issues we work around:
#   (a) OD's setup.py pulls tf-models-official (numpy<2 pin) -> scipy C-ABI crash.
#   (b) JAX + numpy<2 C-ABI mismatch crashes TF startup.
#   (c) Kaggle's system protoc 3.12 emits _pb2.py rejected by modern runtime.
#   (d) Latest grpcio-tools -> protobuf 6.x gencode, incompatible with 5.29.5 rt.
#   (e) Keras 3 removed tf.keras.layers.experimental.SyncBatchNormalization.
#   (f) OD unconditionally imports ssd_efficientnet_bifpn_feature_extractor,
#       which requires 'official' from tf-models-official.
#
# Fix strategy:
#   0. Capture Kaggle's original numpy version.
#   1. Remove JAX.
#   2. Clone tensorflow/models.
#   2b. Patch freezable_sync_batch_norm.py (Keras 3 compat).
#   3. Pinned grpcio-tools + protobuf, regenerate OD protos.
#   4. Install OD API --force-reinstall --no-deps (picks up patched file).
#   5. Ensure OD runtime deps are present.
#   6. Restore numpy + re-pin protobuf.
#   6b. Stub 'official.*' in sys.modules so EfficientDet import path loads.
#   6c. Runtime compatibility shims (estimator + control_flow_ops + purge)
#   7. Verify.

import os
import subprocess


def sh(cmd, tail=None, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if tail:
        out = "\n".join(out.strip().splitlines()[-tail:])
    if out.strip():
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")
    return r.returncode


# ---- Step 0: Capture the current numpy version -----------------------------
r = subprocess.run("pip show numpy | awk '/^Version:/ {print $2}'",
                   shell=True, capture_output=True, text=True)
ORIGINAL_NUMPY = r.stdout.strip()
assert ORIGINAL_NUMPY.startswith("2."), (
    f"Expected numpy 2.x, got '{ORIGINAL_NUMPY}'. Restart the kernel and re-run."
)
print(f">> Captured original numpy version: {ORIGINAL_NUMPY}")

PROTOBUF_VERSION = "5.29.5"
GRPCIO_TOOLS_VERSION = "1.71.0"

# ---- Step 1: Remove JAX ----------------------------------------------------
print("\n>> Removing JAX stack (not needed for OD API training)...")
sh("pip uninstall -y -q jax jaxlib flax optax chex 2>&1", tail=3)

# ---- Step 2: Clone the TensorFlow models repo ------------------------------
TF_MODELS = "/kaggle/working/tf_models"
if not os.path.isdir(os.path.join(TF_MODELS, ".git")):
    print("\n>> Cloning tensorflow/models (shallow)...")
    sh(f"git clone --depth 1 https://github.com/tensorflow/models.git {TF_MODELS} 2>&1",
       check=True, tail=3)
else:
    print("\n>> tensorflow/models already cloned, skipping.")

# ---- Step 2b: Patch for Keras 3 compatibility ------------------------------
print("\n>> Patching freezable_sync_batch_norm.py for Keras 3 compat...")
patch_target = os.path.join(
    TF_MODELS, "research", "object_detection", "core", "freezable_sync_batch_norm.py")
with open(patch_target, "r") as f:
    _src = f.read()
_fixed = _src.replace(
    "tf.keras.layers.experimental.SyncBatchNormalization",
    "tf.keras.layers.BatchNormalization",
)
if _fixed != _src:
    with open(patch_target, "w") as f:
        f.write(_fixed)
    print(f"   Patched {patch_target}.")
else:
    print("   Already patched; skipping.")

# ---- Step 3: Install pinned grpcio-tools + protobuf, regenerate protos -----
print(f"\n>> Installing grpcio-tools=={GRPCIO_TOOLS_VERSION} + protobuf=={PROTOBUF_VERSION}...")
sh(f"pip install -q 'grpcio-tools=={GRPCIO_TOOLS_VERSION}' "
   f"'protobuf=={PROTOBUF_VERSION}' 2>&1", tail=3)

os.chdir(f"{TF_MODELS}/research")

import grpc_tools
GRPC_INC = os.path.join(os.path.dirname(grpc_tools.__file__), "_proto")

print("\n>> Regenerating .proto files with grpc_tools.protoc...")
sh(f"python -m grpc_tools.protoc -I. -I{GRPC_INC} "
   "--python_out=. object_detection/protos/*.proto", check=True)
sh("cp object_detection/packages/tf2/setup.py .", check=True)

# ---- Step 4: Reinstall OD API (picks up patched files) ---------------------
print("\n>> Reinstalling Object Detection API (force, no deps)...")
sh("pip install -q --force-reinstall --no-deps . 2>&1", tail=5)

# ---- Step 5: Ensure OD's runtime dependencies are present ------------------
print("\n>> Ensuring OD runtime dependencies are present...")
sh("pip install -q tf-slim pycocotools pillow lxml matplotlib scipy "
   "Cython six pyparsing tf_keras 2>&1", tail=3)

# ---- Step 6: Restore numpy, re-pin protobuf -------------------------------
print(f"\n>> Restoring numpy=={ORIGINAL_NUMPY} (force-reinstall, no deps)...")
sh(f"pip install -q --force-reinstall --no-deps 'numpy=={ORIGINAL_NUMPY}' 2>&1", tail=3)

print(f"\n>> Re-pinning protobuf=={PROTOBUF_VERSION} (force-reinstall, no deps)...")
sh(f"pip install -q --force-reinstall --no-deps 'protobuf=={PROTOBUF_VERSION}' 2>&1", tail=3)

os.chdir("/kaggle/working")

# ---- Step 6b: Stub 'official' package so EfficientDet import path loads ----
# OD's model_builder unconditionally imports ssd_efficientnet_bifpn_feature_extractor,
# which pulls `from official.legacy.image_classification.efficientnet import
# efficientnet_model`. That module lives in tf-models-official. We don't install
# tf-models-official because its deps would downgrade numpy (see issue (a)).
#
# Instead we register empty package stubs in sys.modules. Python's import
# machinery then resolves the `from ... import efficientnet_model` line to our
# stub. The EfficientDet feature extractor classes reference efficientnet_model
# only inside methods (not at class body), so class definitions succeed.
# We never instantiate those classes (we're training SSD MobileNet V2 FPNLite).
print("\n>> Stubbing 'official.*' in sys.modules (EfficientDet path; unused)...")
import sys
import types


def _stub_package(name):
    """Create/upgrade a package stub in sys.modules and wire parent attrs."""
    if name not in sys.modules:
        pkg = types.ModuleType(name)
        pkg.__path__ = []                       # Mark as package.
        sys.modules[name] = pkg
    if '.' in name:
        parent_name, leaf = name.rsplit('.', 1)
        setattr(sys.modules[parent_name], leaf, sys.modules[name])


def _stub_module(name):
    """Create a leaf module stub and wire parent attr."""
    if name not in sys.modules:
        sys.modules[name] = types.ModuleType(name)
    parent_name, leaf = name.rsplit('.', 1)
    setattr(sys.modules[parent_name], leaf, sys.modules[name])


for _name in [
    'official',
    'official.legacy',
    'official.legacy.image_classification',
    'official.legacy.image_classification.efficientnet',
    'official.vision',
    'official.vision.image_classification',
    'official.vision.image_classification.efficientnet',
    'official.modeling',
    'official.modeling.optimization',
]:
    _stub_package(_name)
for _name in [
    'official.legacy.image_classification.efficientnet.efficientnet_model',
    'official.vision.image_classification.efficientnet.efficientnet_model',
    'official.modeling.optimization.ema_optimizer',
]:
    _stub_module(_name)

# ---- Step 6c: Runtime compatibility shims (TF 2.19 / Keras 3) ------------
#
# Three families of breakage addressed here:
#   1. tf.estimator / tf.compat.v1.estimator removed -> stub ModeKeys.
#   2. Many tf.python.ops.* internals moved in TF 2.16+ -> alias the legacy
#      names back onto the old internal module (tf_slim uses them).
#   3. object_detection.inputs may have been partially imported in a failed
#      attempt; purge so the next import is clean.

# import sys (redundant already imported at Step 6b above)
# import types (redundant already imported at Step 6b above)
import tensorflow as tf

# ---- Shim 1: tf.estimator.ModeKeys ----------------------------------------
class _ModeKeys:
    TRAIN = 'train'
    EVAL = 'eval'
    PREDICT = 'infer'


for _mod_name, _attr_on_tf in [
    ('tensorflow.compat.v1.estimator', ('compat', 'v1', 'estimator')),
    ('tensorflow.estimator', ('estimator',)),
]:
    _stub = types.ModuleType(_mod_name)
    _stub.ModeKeys = _ModeKeys
    sys.modules[_mod_name] = _stub
    # Wire the stub onto the tf module tree too (so `tf.compat.v1.estimator.X`
    # attribute access works, not just `from ... import estimator`).
    _obj = tf
    for _p in _attr_on_tf[:-1]:
        _obj = getattr(_obj, _p)
    setattr(_obj, _attr_on_tf[-1], _stub)

print("Registered tf.estimator / tf.compat.v1.estimator ModeKeys stubs.")


# ---- Shim 2: tf.python.ops.control_flow_ops legacy attribute aliases ------
# tf_slim's tfexample_decoder imports control_flow_ops and calls .case, .cond,
# etc. TF 2.16+ split these into sibling modules and removed the re-exports.
# The public tf.<name> equivalents still exist, so we alias them back.
from tensorflow.python.ops import control_flow_ops as _cfo

_missing = []
for _name in ['case', 'cond', 'group', 'no_op', 'tuple', 'while_loop']:
    if not hasattr(_cfo, _name):
        _public = getattr(tf, _name, None)
        if _public is not None:
            setattr(_cfo, _name, _public)
            _missing.append(_name)

if _missing:
    print(f"Patched control_flow_ops: {', '.join(_missing)}")
else:
    print("control_flow_ops already has all expected attributes; nothing to patch.")


# ---- Shim 3: purge any half-imported OD modules from the previous failure -
# When a `from X import Y` crashes mid-import, Python usually rolls back, but
# autograph-converted functions can leave cached traces that still reference
# the old (unpatched) control_flow_ops. Clearing these forces a fresh import.
for _m in list(sys.modules):
    if _m.startswith('object_detection.inputs') \
       or _m.startswith('object_detection.builders.dataset_builder') \
       or _m.startswith('object_detection.data_decoders') \
       or _m.startswith('tf_slim.data'):
        del sys.modules[_m]
print("Purged stale object_detection/tf_slim data pipeline modules.")

# ---- Step 7: Verify --------------------------------------------------------
print("\n>> Verifying imports...")
import numpy as np
print(f"   numpy:      {np.__version__}")
assert np.__version__ == ORIGINAL_NUMPY, "numpy version mismatch after restore!"

import google.protobuf
print(f"   protobuf:   {google.protobuf.__version__}")
assert google.protobuf.__version__ == PROTOBUF_VERSION, "protobuf version drifted!"

import tensorflow as tf
print(f"   tensorflow: {tf.__version__}")
print(f"   keras:      {tf.keras.__version__}")
print(f"   GPUs:       {tf.config.list_physical_devices('GPU')}")

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

from object_detection.utils import config_util
from object_detection.utils import label_map_util
from object_detection.builders import model_builder

print("\n✓ Environment ready: TF, OD API (patched + stubbed), numpy+protobuf pinned, 2× GPU.")

>> Captured original numpy version: 2.0.2

>> Removing JAX stack (not needed for OD API training)...

>> Cloning tensorflow/models (shallow)...
Cloning into '/kaggle/working/tf_models'...

>> Patching freezable_sync_batch_norm.py for Keras 3 compat...
   Patched /kaggle/working/tf_models/research/object_detection/core/freezable_sync_batch_norm.py.

>> Installing grpcio-tools==1.71.0 + protobuf==5.29.5...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.8 MB/s eta 0:00:00

>> Regenerating .proto files with grpc_tools.protoc...

>> Reinstalling Object Detection API (force, no deps)...

>> Ensuring OD runtime dependencies are present...

>> Restoring numpy==2.0.2 (force-reinstall, no deps)...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 82.4 MB/s eta 0:00:00

>> Re-pinning protobuf==5.29.5 (force-reinstall, no deps)...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 16.2 MB/s eta 0:00:00



2026-04-25 16:19:39.006029: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777133979.413508      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777133979.526245      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777133980.601770      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777133980.601823      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777133980.601826      55 computation_placer.cc:177] computation placer alr

Registered tf.estimator / tf.compat.v1.estimator ModeKeys stubs.
Patched control_flow_ops: case, cond, while_loop
Purged stale object_detection/tf_slim data pipeline modules.

>> Verifying imports...
   numpy:      2.0.2
   protobuf:   5.29.5
   tensorflow: 2.19.0
   keras:      3.10.0
   GPUs:       [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl5mutex6unlockEv']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZN3tsl7strings13safe_strtou64ESt17basic_string_viewIcSt11char_traitsIcEEPm']
  warnings.warn(


✓ Environment ready: TF, OD API (patched + stubbed), numpy+protobuf pinned, 2× GPU.


In [2]:
# Cell #2: Sanity check. Re-running this after a kernel restart without running
# cell #1 first will fail — the 'official.*' sys.modules stubs from cell #1
# don't persist across kernel restarts.
import numpy as np
import tensorflow as tf
import google.protobuf

from object_detection.builders import model_builder  # noqa: F401
from object_detection.utils import config_util        # noqa: F401
from object_detection.utils import label_map_util     # noqa: F401

assert np.__version__ == "2.0.2", f"numpy drifted to {np.__version__}"
assert google.protobuf.__version__ == "5.29.5", \
    f"protobuf drifted to {google.protobuf.__version__}"
assert len(tf.config.list_physical_devices('GPU')) >= 1, "No GPU visible to TF"

print(f"OK — numpy {np.__version__}, protobuf {google.protobuf.__version__}, "
      f"TF {tf.__version__}, keras {tf.keras.__version__}, "
      f"{len(tf.config.list_physical_devices('GPU'))}× GPU, OD API importable.")

OK — numpy 2.0.2, protobuf 5.29.5, TF 2.19.0, keras 3.10.0, 2× GPU, OD API importable.


## 2. Locate Training Data

In [3]:
# Auto-detect paths from Notebook 1 output.
import os
import json
import glob
from pathlib import Path

DATA_DIR = None
METADATA = None

# Search for our dataset metadata anywhere under /kaggle/input.
# Notebook 1's output is attached via the Notebooks tab; it typically lands
# under /kaggle/input/<notebook-slug>/ but we don't hardcode that.
for meta_path in Path("/kaggle/input").rglob("dataset_metadata.json"):
    with open(meta_path) as f:
        METADATA = json.load(f)
    DATA_DIR = str(meta_path.parent)
    break

if DATA_DIR is None:
    # Print what IS under /kaggle/input so we can see whether Notebook 1's
    # output was actually attached and where it landed.
    print("dataset_metadata.json not found. Top-level contents of /kaggle/input:")
    for entry in sorted(Path("/kaggle/input").iterdir()):
        print(f"  {entry}")
    raise FileNotFoundError(
        "Attach Notebook 1 (Data Preparation) as input via Add Input → "
        "Notebooks tab, making sure the saved notebook version has outputs."
    )

# Resolve paths.
TFRECORD_DIR = os.path.join(DATA_DIR, "tfrecords")
LABEL_MAP_PATH = os.path.join(DATA_DIR, "label_map.pbtxt")

# Verify files exist.
train_records = sorted(glob.glob(os.path.join(TFRECORD_DIR, "train-*.tfrecord")))
val_records = sorted(glob.glob(os.path.join(TFRECORD_DIR, "val-*.tfrecord")))

print(f"Data directory:    {DATA_DIR}")
print(f"TFRecord dir:      {TFRECORD_DIR}")
print(f"Label map:         {LABEL_MAP_PATH}")
print(f"Train shards:      {len(train_records)}")
print(f"Val shards:        {len(val_records)}")
print(f"\nDataset metadata:")
print(json.dumps(METADATA, indent=2))

assert len(train_records) > 0, \
    f"No train TFRecords under {TFRECORD_DIR}. Contents: {os.listdir(TFRECORD_DIR)}"
assert len(val_records) > 0, \
    f"No val TFRecords under {TFRECORD_DIR}. Contents: {os.listdir(TFRECORD_DIR)}"
assert os.path.exists(LABEL_MAP_PATH), f"Label map missing at {LABEL_MAP_PATH}"
print("\nAll data verified.")

Data directory:    /kaggle/input/notebooks/giladfaibish/data-preparation/snipeit_person_dataset
TFRecord dir:      /kaggle/input/notebooks/giladfaibish/data-preparation/snipeit_person_dataset/tfrecords
Label map:         /kaggle/input/notebooks/giladfaibish/data-preparation/snipeit_person_dataset/label_map.pbtxt
Train shards:      16
Val shards:        4

Dataset metadata:
{
  "project": "Smart Sniper Spotter - Target Detection Module",
  "description": "Unified person detection dataset (COCO outdoor + WiderPerson)",
  "created_with": "Notebook 1 - Data Preparation",
  "num_classes": 1,
  "class_name": "person",
  "train_examples": 65717,
  "val_examples": 7301,
  "train_shards": 16,
  "val_shards": 4,
  "sources": {
    "coco_outdoor_person": 64018,
    "widerperson": 9000
  },
  "train_source_breakdown": {
    "coco": 57657,
    "widerperson": 8060
  },
  "val_source_breakdown": {
    "coco": 6361,
    "widerperson": 940
  },
  "label_map_path": "label_map.pbtxt",
  "tfrecord_dir": "

## 3. Download Pretrained Checkpoint

We use the COCO-pretrained SSD MobileNet V2 FPNLite 640×640 checkpoint 
from the TensorFlow 2 Model Zoo. This gives us a strong starting point 
with learned feature representations for person detection.


In [4]:
WORK_DIR = "/kaggle/working"
MODEL_DIR = os.path.join(WORK_DIR, "pretrained_model")
TRAIN_DIR = os.path.join(WORK_DIR, "training_output")
EXPORT_DIR = os.path.join(WORK_DIR, "exported_model")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TRAIN_DIR, exist_ok=True)

# TF2 Model Zoo URL for SSD MobileNet V2 FPNLite 640x640
MODEL_URL = "http://download.tensorflow.org/models/object_detection/tf2/20200711/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8.tar.gz"
MODEL_TAR = os.path.join(WORK_DIR, "ssd_mobilenet_v2_fpnlite_640x640.tar.gz")
MODEL_NAME = "ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8"

PRETRAINED_CKPT_DIR = os.path.join(MODEL_DIR, MODEL_NAME)

if os.path.exists(os.path.join(PRETRAINED_CKPT_DIR, "checkpoint")):
    print(f"Pretrained model already exists at {PRETRAINED_CKPT_DIR}")
else:
    print("Downloading pretrained SSD MobileNet V2 FPNLite 640x640...")
    !wget -q --show-progress -O "{MODEL_TAR}" "{MODEL_URL}"
    
    print("Extracting...")
    !tar -xf "{MODEL_TAR}" -C "{MODEL_DIR}"
    
    # Clean up tar file
    os.remove(MODEL_TAR)
    print("Done.")

# Verify checkpoint
ckpt_prefix = os.path.join(PRETRAINED_CKPT_DIR, "checkpoint", "ckpt-0")
print(f"\nCheckpoint directory: {PRETRAINED_CKPT_DIR}")
!ls -la "{PRETRAINED_CKPT_DIR}/checkpoint/"


/kaggle/working/ssd 100%[===================>]  19.57M  89.7MB/s    in 0.2s    
Extracting...
Done.

Checkpoint directory: /kaggle/working/pretrained_model/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8
total 10524
drwxr-x--- 2 345018 89939     4096 Jul 10  2020 .
drwxr-x--- 4 345018 89939     4096 Jul 11  2020 ..
-rw-r----- 1 345018 89939      166 Jul 10  2020 checkpoint
-rw-r----- 1 345018 89939 10752177 Jul 10  2020 ckpt-0.data-00000-of-00001
-rw-r----- 1 345018 89939     6872 Jul 10  2020 ckpt-0.index


## 4. Configure Training Pipeline

The pipeline config defines:
- **Model architecture** (SSD MobileNet V2 FPNLite)
- **Input size** (640×640)
- **Number of classes** (1 — person)
- **Training parameters** (batch size, learning rate, steps)
- **Data augmentation** pipeline tailored to our operational conditions
- **Quantization-aware training** for EdgeTPU compatibility

### Augmentation Summary
| Augmentation | Purpose | Parameters |
|---|---|---|
| Random horizontal flip | General robustness | 50% probability |
| Random crop | Scale variation, partial person | min area 0.5 |
| Random brightness/contrast | Dusk/dawn lighting | ±0.15 brightness, contrast 0.8–1.2 |
| Random RGB-to-gray | Robustness to color shift | 10% probability |
| Random image scale | Small person simulation | scales 0.5–2.0 |
| SSD random crop | Standard SSD augmentation | Various IoU thresholds |


In [5]:
# Training hyperparameters
BATCH_SIZE = 8           # Fits in T4 16GB VRAM for 640x640
NUM_STEPS = 30000        # ~30K steps; adjust based on convergence
WARMUP_STEPS = 1000
NUM_CLASSES = 1

# Learning rate schedule
BASE_LEARNING_RATE = 0.01
WARMUP_LEARNING_RATE = 0.001

# Checkpoint frequency
CHECKPOINT_EVERY_N = 1000

# TFRecord patterns
TRAIN_TFRECORD_PATTERN = os.path.join(TFRECORD_DIR, "train-*.tfrecord")
VAL_TFRECORD_PATTERN = os.path.join(TFRECORD_DIR, "val-*.tfrecord")

# Fine-tune checkpoint path
FINE_TUNE_CKPT = os.path.join(PRETRAINED_CKPT_DIR, "checkpoint", "ckpt-0")

print(f"Batch size:        {BATCH_SIZE}")
print(f"Training steps:    {NUM_STEPS}")
print(f"Checkpoint every:  {CHECKPOINT_EVERY_N} steps")
print(f"Base LR:           {BASE_LEARNING_RATE}")
print(f"Fine-tune from:    {FINE_TUNE_CKPT}")


Batch size:        8
Training steps:    30000
Checkpoint every:  1000 steps
Base LR:           0.01
Fine-tune from:    /kaggle/working/pretrained_model/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint/ckpt-0


In [6]:
# Write the pipeline config file
# This is based on the default SSD MobileNet V2 FPNLite 640x640 config
# with modifications for single-class person detection and our augmentation strategy

PIPELINE_CONFIG_PATH = os.path.join(WORK_DIR, "pipeline.config")

pipeline_config = f"""
model {{
  ssd {{
    num_classes: {NUM_CLASSES}
    image_resizer {{
      fixed_shape_resizer {{
        height: 640
        width: 640
      }}
    }}
    feature_extractor {{
      type: "ssd_mobilenet_v2_fpn_keras"
      depth_multiplier: 1.0
      min_depth: 16
      conv_hyperparams {{
        regularizer {{
          l2_regularizer {{
            weight: 0.00004
          }}
        }}
        initializer {{
          random_normal_initializer {{
            mean: 0.0
            stddev: 0.01
          }}
        }}
        activation: RELU_6
        batch_norm {{
          decay: 0.997
          scale: true
          epsilon: 0.001
        }}
      }}
      use_depthwise: true
      override_base_feature_extractor_hyperparams: true
      fpn {{
        min_level: 3
        max_level: 7
        additional_layer_depth: 128
      }}
    }}
    box_coder {{
      faster_rcnn_box_coder {{
        y_scale: 10.0
        x_scale: 10.0
        height_scale: 5.0
        width_scale: 5.0
      }}
    }}
    matcher {{
      argmax_matcher {{
        matched_threshold: 0.5
        unmatched_threshold: 0.5
        ignore_thresholds: false
        negatives_lower_than_unmatched: true
        force_match_for_each_row: true
        use_matmul_gather: true
      }}
    }}
    similarity_calculator {{
      iou_similarity {{
      }}
    }}
    box_predictor {{
      weight_shared_convolutional_box_predictor {{
        conv_hyperparams {{
          regularizer {{
            l2_regularizer {{
              weight: 0.00004
            }}
          }}
          initializer {{
            random_normal_initializer {{
              mean: 0.0
              stddev: 0.01
            }}
          }}
          activation: RELU_6
          batch_norm {{
            decay: 0.997
            scale: true
            epsilon: 0.001
          }}
        }}
        depth: 128
        num_layers_before_predictor: 4
        kernel_size: 3
        class_prediction_bias_init: -4.6
      }}
    }}
    anchor_generator {{
      multiscale_anchor_generator {{
        min_level: 3
        max_level: 7
        anchor_scale: 4.0
        aspect_ratios: 1.0
        aspect_ratios: 2.0
        aspect_ratios: 0.5
        scales_per_octave: 2
      }}
    }}
    post_processing {{
      batch_non_max_suppression {{
        score_threshold: 0.01
        iou_threshold: 0.6
        max_detections_per_class: 20
        max_total_detections: 20
        use_static_shapes: false
      }}
      score_converter: SIGMOID
    }}
    normalize_loss_by_num_matches: true
    loss {{
      localization_loss {{
        weighted_smooth_l1 {{
        }}
      }}
      classification_loss {{
        weighted_sigmoid_focal {{
          gamma: 2.0
          alpha: 0.25
        }}
      }}
      classification_weight: 1.0
      localization_weight: 1.0
    }}
    encode_background_as_zeros: true
    normalize_loc_loss_by_codesize: true
    inplace_batchnorm_update: true
    freeze_batchnorm: false
  }}
}}

train_config {{
  batch_size: {BATCH_SIZE}
  data_augmentation_options {{
    random_horizontal_flip {{
    }}
  }}
  data_augmentation_options {{
    random_crop_image {{
      min_object_covered: 0.0
      min_aspect_ratio: 0.75
      max_aspect_ratio: 3.0
      min_area: 0.5
      max_area: 1.0
      overlap_thresh: 0.0
    }}
  }}
  data_augmentation_options {{
    ssd_random_crop {{
    }}
  }}
  data_augmentation_options {{
    random_adjust_brightness {{
      max_delta: 0.15
    }}
  }}
  data_augmentation_options {{
    random_adjust_contrast {{
      min_delta: 0.8
      max_delta: 1.2
    }}
  }}
  data_augmentation_options {{
    random_rgb_to_gray {{
      probability: 0.1
    }}
  }}
  data_augmentation_options {{
    random_image_scale {{
      min_scale_ratio: 0.5
      max_scale_ratio: 2.0
    }}
  }}
  sync_replicas: true
  optimizer {{
    momentum_optimizer {{
      learning_rate {{
        cosine_decay_learning_rate {{
          learning_rate_base: {BASE_LEARNING_RATE}
          total_steps: {NUM_STEPS}
          warmup_learning_rate: {WARMUP_LEARNING_RATE}
          warmup_steps: {WARMUP_STEPS}
        }}
      }}
      momentum_optimizer_value: 0.9
    }}
    use_moving_average: false
  }}
  fine_tune_checkpoint: "{FINE_TUNE_CKPT}"
  num_steps: {NUM_STEPS}
  startup_delay_steps: 0.0
  replicas_to_aggregate: 8
  max_number_of_boxes: 100
  unpad_groundtruth_tensors: false
  fine_tune_checkpoint_type: "detection"
  fine_tune_checkpoint_version: V2
}}

train_input_reader {{
  label_map_path: "{LABEL_MAP_PATH}"
  tf_record_input_reader {{
    input_path: "{TRAIN_TFRECORD_PATTERN}"
  }}
}}

eval_config {{
  metrics_set: "coco_detection_metrics"
  use_moving_averages: false
  batch_size: 1
}}

eval_input_reader {{
  label_map_path: "{LABEL_MAP_PATH}"
  shuffle: false
  num_epochs: 1
  tf_record_input_reader {{
    input_path: "{VAL_TFRECORD_PATTERN}"
  }}
}}
"""

with open(PIPELINE_CONFIG_PATH, "w") as f:
    f.write(pipeline_config)

print(f"Pipeline config written to: {PIPELINE_CONFIG_PATH}")
print(f"Config size: {len(pipeline_config)} chars")


Pipeline config written to: /kaggle/working/pipeline.config
Config size: 5101 chars


## 5. Verify Pipeline Configuration

In [7]:
# Parse and verify the config
from object_detection.utils import config_util

configs = config_util.get_configs_from_pipeline_file(PIPELINE_CONFIG_PATH)
model_config = configs['model']
train_config = configs['train_config']
train_input_config = configs['train_input_config']
eval_config = configs['eval_config']
eval_input_config = configs['eval_input_config']

print("Pipeline config verified successfully.")
print(f"  Model type:       {model_config.WhichOneof('model')}")
print(f"  Num classes:      {model_config.ssd.num_classes}")
print(f"  Batch size:       {train_config.batch_size}")
print(f"  Num steps:        {train_config.num_steps}")
print(f"  Fine-tune ckpt:   {train_config.fine_tune_checkpoint}")
print(f"  Fine-tune type:   {train_config.fine_tune_checkpoint_type}")
print(f"  Augmentations:    {len(train_config.data_augmentation_options)} configured")
for i, aug in enumerate(train_config.data_augmentation_options):
    aug_type = aug.WhichOneof('preprocessing_step')
    print(f"    {i+1}. {aug_type}")

# Quick data pipeline test — load one batch
print("\nTesting data pipeline (loading 1 batch)...")
from object_detection.inputs import train_input
from object_detection import inputs

train_dataset = inputs.train_input(
    train_config=train_config,
    train_input_config=train_input_config,
    model_config=model_config,
    model=None
)

for batch in train_dataset.take(1):
    images = batch[0]['image']
    print(f"  Batch shape: {images.shape}")
    print(f"  Dtype: {images.dtype}")
    print(f"  Value range: [{tf.reduce_min(images).numpy():.2f}, {tf.reduce_max(images).numpy():.2f}]")
    print("  Data pipeline OK!")
    break


Pipeline config verified successfully.
  Model type:       ssd
  Num classes:      1
  Batch size:       8
  Num steps:        30000
  Fine-tune ckpt:   /kaggle/working/pretrained_model/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint/ckpt-0
  Fine-tune type:   detection
  Augmentations:    7 configured
    1. random_horizontal_flip
    2. random_crop_image
    3. ssd_random_crop
    4. random_adjust_brightness
    5. random_adjust_contrast
    6. random_rgb_to_gray
    7. random_image_scale

Testing data pipeline (loading 1 batch)...
INFO:tensorflow:Reading unweighted datasets: ['/kaggle/input/notebooks/giladfaibish/data-preparation/snipeit_person_dataset/tfrecords/train-*.tfrecord']
INFO:tensorflow:Reading record datasets for input file: ['/kaggle/input/notebooks/giladfaibish/data-preparation/snipeit_person_dataset/tfrecords/train-*.tfrecord']
INFO:tensorflow:Number of filenames to read: 16
Instructions for updating:
Use `tf.data.Dataset.interleave(map_func, cycle_length, blo

I0000 00:00:1777134018.243412      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777134018.249520      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Instructions for updating:
Create a `tf.sparse.SparseTensor` and use `tf.sparse.to_dense` instead.
Instructions for updating:
`seed2` arg is deprecated.Use sample_distorted_bounding_box_v2 instead.
Instructions for updating:
keep_dims is deprecated, use keepdims instead
Instructions for updating:
Use `tf.cast` instead.
  Batch shape: (8, 640, 640, 3)
  Dtype: <dtype: 'float32'>
  Value range: [-1.00, 1.00]
  Data pipeline OK!


## 6. Train the Model

Training uses the TF2 Object Detection API's `model_main_tf2.py` script.
Checkpoints are saved every 1000 steps so you can resume if the session ends.

**Expected training time:** ~3–5 hours on T4 GPU for 30K steps.

### Resume from previous session
If your session was interrupted, simply re-run this cell. The training script 
automatically detects existing checkpoints in the output directory and resumes 
from the latest one.


In [8]:
# Check for existing checkpoints (session resume)
existing_ckpts = glob.glob(os.path.join(TRAIN_DIR, "ckpt-*.index"))
if existing_ckpts:
    latest_step = max(
        int(os.path.basename(c).split("-")[1].split(".")[0]) 
        for c in existing_ckpts
    )
    print(f"Found existing checkpoints. Latest: step {latest_step}")
    print(f"Training will RESUME from step {latest_step}.")
else:
    print("No existing checkpoints found. Training from pretrained COCO checkpoint.")

print(f"\nOutput directory: {TRAIN_DIR}")
print(f"Target steps:     {NUM_STEPS}")


No existing checkpoints found. Training from pretrained COCO checkpoint.

Output directory: /kaggle/working/training_output
Target steps:     30000


In [20]:
# Cell #8.5: Write a wrapper script that applies TF 2.19 / Keras 3 compat
# shims in the training subprocess, then delegates to model_main_tf2.main.
#
# Necessary because cell #9 launches training as `!python ...` (subprocess),
# which doesn't inherit the kernel's sys.modules patches from cell #1's Step 6c.

WRAPPER_PATH = "/kaggle/working/train_with_shims.py"

_wrapper_src = r'''"""
Launcher for object_detection.model_main_tf2 with TF 2.19 / Keras 3 compat
shims and Keras 2 (via tf_keras) routing for the OD code paths.
"""
# Must precede any tensorflow import. Causes tf.keras to resolve to tf_keras
# (Keras 2.x), which is what the Object Detection API was written against.
# Avoids: MobileNetV2 'layers' kwarg removal, set_learning_phase removal, and
# other Keras 3 API drops we've been patching one-by-one.
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import sys
import types


# ---------------------------------------------------------------------------
# Shim A: sys.modules package stubs. Must be installed BEFORE any import of
# tensorflow or object_detection, so the import machinery finds them.
# ---------------------------------------------------------------------------
def _stub_package(name):
    """Register an empty package in sys.modules and wire it onto its parent."""
    if name not in sys.modules:
        pkg = types.ModuleType(name)
        pkg.__path__ = []
        sys.modules[name] = pkg
    if '.' in name:
        parent, leaf = name.rsplit('.', 1)
        setattr(sys.modules[parent], leaf, sys.modules[name])


def _stub_module(name):
    """Register an empty leaf module in sys.modules and wire it onto its parent."""
    if name not in sys.modules:
        sys.modules[name] = types.ModuleType(name)
    parent, leaf = name.rsplit('.', 1)
    setattr(sys.modules[parent], leaf, sys.modules[name])


# 'official.*' -- modules from tf-models-official that OD imports at load time.
# We don't install tf-models-official because its deps cascade to numpy<2,
# which breaks scipy/JAX C-ABI on TF startup. None of the code paths that
# actually USE these are exercised by SSD MobileNet V2 FPNLite + COCO eval:
#   - official.legacy/vision.image_classification.efficientnet -- EfficientDet
#     feature extractor (we use SSD).
#   - official.modeling.optimization.ema_optimizer -- EMA optimizer (our
#     pipeline config doesn't enable optimizer.ema_decay).
for _n in [
    'official',
    'official.legacy',
    'official.legacy.image_classification',
    'official.legacy.image_classification.efficientnet',
    'official.vision',
    'official.vision.image_classification',
    'official.vision.image_classification.efficientnet',
    'official.modeling',
    'official.modeling.optimization',
]:
    _stub_package(_n)
for _n in [
    'official.legacy.image_classification.efficientnet.efficientnet_model',
    'official.vision.image_classification.efficientnet.efficientnet_model',
    'official.modeling.optimization.ema_optimizer',
]:
    _stub_module(_n)

# 'lvis' -- LVIS dataset evaluator. OD imports metrics.lvis_evaluation +
# metrics.lvis_tools at load time even though we use COCO evaluation.
# lvis_tools.py subclasses lvis.LVIS / LVISEval / LVISResults at module scope,
# so the stub must expose those names as actual classes (not just modules).
# Methods on the subclasses are never called -- we evaluate with COCO.
_stub_package('lvis')
_stub_module('lvis.lvis')
_stub_module('lvis.eval')
_stub_module('lvis.results')


class _LVISStub:
    """Placeholder base class for OD's LVIS subclasses; never instantiated."""
    def __init__(self, *args, **kwargs):
        raise RuntimeError(
            "lvis is stubbed out; this code path is not used in COCO-only training."
        )


# Place the names where lvis_tools looks them up (`lvis.LVIS`, etc.) AND on
# the inner submodules in case anything qualifies as `lvis.lvis.LVIS`.
sys.modules['lvis'].LVIS = _LVISStub
sys.modules['lvis'].LVISEval = _LVISStub
sys.modules['lvis'].LVISResults = _LVISStub
sys.modules['lvis.lvis'].LVIS = _LVISStub
sys.modules['lvis.eval'].LVISEval = _LVISStub
sys.modules['lvis.results'].LVISResults = _LVISStub

# ---------------------------------------------------------------------------
# Shim B: tf.estimator / tf.compat.v1.estimator (removed in TF 2.16+).
# OD only needs ModeKeys string constants.
# ---------------------------------------------------------------------------
import tensorflow as tf


class _ModeKeys:
    TRAIN = 'train'
    EVAL = 'eval'
    PREDICT = 'infer'


for _mod_name, _attr_path in [
    ('tensorflow.compat.v1.estimator', ('compat', 'v1', 'estimator')),
    ('tensorflow.estimator', ('estimator',)),
]:
    _stub = types.ModuleType(_mod_name)
    _stub.ModeKeys = _ModeKeys
    sys.modules[_mod_name] = _stub
    _obj = tf
    for _p in _attr_path[:-1]:
        _obj = getattr(_obj, _p)
    setattr(_obj, _attr_path[-1], _stub)


# ---------------------------------------------------------------------------
# Shim C: tf.python.ops.control_flow_ops legacy attribute aliases (tf_slim
# calls .case/.cond/etc on this internal module; TF 2.16+ moved them out).
# ---------------------------------------------------------------------------
from tensorflow.python.ops import control_flow_ops as _cfo

for _name in ['case', 'cond', 'group', 'no_op', 'tuple', 'while_loop']:
    if not hasattr(_cfo, _name):
        _public = getattr(tf, _name, None)
        if _public is not None:
            setattr(_cfo, _name, _public)


# ---------------------------------------------------------------------------
# Shim D: Keras 3 removed tf.keras.backend.set_learning_phase (and other
# legacy globals). OD's model_lib_v2._dummy_computation_fn calls
# set_learning_phase(False) when building the model for checkpoint restore.
#
# In Keras 2, this set a global train/eval flag. In Keras 3, mode is set
# per-call via training=True/False kwargs, so the call is meaningless.
# Replacing it with a no-op preserves correct behavior because OD's actual
# training and eval paths pass `training=` explicitly.
# ---------------------------------------------------------------------------
if not hasattr(tf.keras.backend, 'set_learning_phase'):
    tf.keras.backend.set_learning_phase = lambda *args, **kwargs: None
if not hasattr(tf.keras.backend, 'learning_phase'):
    # Some OD eval paths read the flag back. Returning 0 (=eval) is the
    # historical default and matches OD's intended use here.
    tf.keras.backend.learning_phase = lambda: 0

# ---------------------------------------------------------------------------
# Delegate to model_main_tf2. Importing it registers all its absl flags;
# app.run(main) then parses sys.argv and calls main(argv).
# ---------------------------------------------------------------------------
from absl import app
from object_detection import model_main_tf2  # noqa: E402

if __name__ == '__main__':
    app.run(model_main_tf2.main)
'''

with open(WRAPPER_PATH, "w") as f:
    f.write(_wrapper_src)

print(f"Wrote wrapper: {WRAPPER_PATH} ({len(_wrapper_src)} chars)")

Wrote wrapper: /kaggle/working/train_with_shims.py (6546 chars)


In [21]:
# Launch training via the compat wrapper (see cell #8.5).
import sys

print(f"Training script: {WRAPPER_PATH}")
print(f"\nStarting training...")
print(f"{'='*60}")

!python "{WRAPPER_PATH}" \
    --pipeline_config_path="{PIPELINE_CONFIG_PATH}" \
    --model_dir="{TRAIN_DIR}" \
    --checkpoint_every_n={CHECKPOINT_EVERY_N} \
    --num_workers=2 \
    --alsologtostderr

Training script: /kaggle/working/train_with_shims.py

Starting training...
2026-04-25 16:52:48.844055: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777135968.866109     578 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777135968.872958     578 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777135968.889370     578 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777135968.889423     578 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:

## 7. Training Results

In [22]:
# List checkpoints
print("Saved checkpoints:")
ckpt_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "ckpt-*.index")))
for ckpt in ckpt_files:
    step = os.path.basename(ckpt).split("-")[1].split(".")[0]
    size_mb = os.path.getsize(ckpt.replace(".index", ".data-00000-of-00001")) / (1024*1024)
    print(f"  Step {step}: {size_mb:.1f} MB")

# Check for latest checkpoint
if ckpt_files:
    latest = ckpt_files[-1]
    latest_step = os.path.basename(latest).split("-")[1].split(".")[0]
    print(f"\nLatest checkpoint: step {latest_step}")
else:
    print("\nNo checkpoints found. Training may have failed.")
    print("Check the training output above for errors.")


Saved checkpoints:
  Step 25: 28.6 MB
  Step 26: 28.6 MB
  Step 27: 28.6 MB
  Step 28: 28.6 MB
  Step 29: 28.6 MB
  Step 30: 28.6 MB
  Step 31: 28.6 MB

Latest checkpoint: step 31


## 8. Evaluate on Validation Set

Run evaluation to get COCO detection metrics (mAP, AR) on the validation split.


In [24]:
# Run evaluation on the validation set using the compat wrapper from cell #8.5.
# Setting --checkpoint_dir tells model_main_tf2 to switch from training to
# eval-only mode: it loads the most recent checkpoint from TRAIN_DIR and runs
# the COCO evaluator across the val TFRecords once.

print("Running evaluation on validation set...")
print("=" * 60)

!python "{WRAPPER_PATH}" \
    --pipeline_config_path="{PIPELINE_CONFIG_PATH}" \
    --model_dir="{TRAIN_DIR}" \
    --checkpoint_dir="{TRAIN_DIR}" \
    --alsologtostderr

print("\n" + "=" * 60)
print("Evaluation complete. Check metrics above.")
print("Key metrics to look for:")
print("  - DetectionBoxes_Precision/mAP         (target: ≥ 0.80)")
print("  - DetectionBoxes_Recall/AR@100         (target: ≥ 0.90)")
print("  - DetectionBoxes_Precision/mAP (small) (critical for distant targets)")

Running evaluation on validation set...
2026-04-25 19:17:10.988827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777144631.011283   90430 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777144631.018697   90430 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777144631.039196   90430 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777144631.039249   90430 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777144631.039257   90430 computati

## 9. Export to TFLite (INT8 for EdgeTPU)

Export the trained model to:
1. **SavedModel** format (standard TF2)
2. **TFLite INT8** quantized model (for EdgeTPU compilation)
3. **EdgeTPU-compiled** model (if edgetpu_compiler is available)


In [26]:
# Cell #11.5: Write a wrapper script for exporter_main_v2.py.
# Same pattern as cell #8.5: subprocess invocations don't inherit the kernel's
# sys.modules shims, so the exporter crashes on `from object_detection...`
# without our compat layer applied first.

EXPORTER_WRAPPER_PATH = "/kaggle/working/export_with_shims.py"

_exporter_wrapper_src = r'''"""
Launcher for object_detection.exporter_main_v2 with TF 2.19 / Keras 3 compat
shims. Accepts the same CLI flags as exporter_main_v2.py.
"""
# Must precede any tensorflow import. Causes tf.keras to resolve to tf_keras
# (Keras 2.x), which is what OD's exporter and saved_model code expect.
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import sys
import types


# ---------------------------------------------------------------------------
# Shim A: sys.modules package stubs. Must run BEFORE any tensorflow or
# object_detection import so the import machinery resolves them.
# ---------------------------------------------------------------------------
def _stub_package(name):
    if name not in sys.modules:
        pkg = types.ModuleType(name)
        pkg.__path__ = []
        sys.modules[name] = pkg
    if '.' in name:
        parent, leaf = name.rsplit('.', 1)
        setattr(sys.modules[parent], leaf, sys.modules[name])


def _stub_module(name):
    if name not in sys.modules:
        sys.modules[name] = types.ModuleType(name)
    parent, leaf = name.rsplit('.', 1)
    setattr(sys.modules[parent], leaf, sys.modules[name])


# 'official.*' -- modules from tf-models-official OD imports at load time but
# doesn't actually use for SSD MobileNet V2 FPNLite export.
for _n in [
    'official',
    'official.legacy',
    'official.legacy.image_classification',
    'official.legacy.image_classification.efficientnet',
    'official.vision',
    'official.vision.image_classification',
    'official.vision.image_classification.efficientnet',
    'official.modeling',
    'official.modeling.optimization',
]:
    _stub_package(_n)
for _n in [
    'official.legacy.image_classification.efficientnet.efficientnet_model',
    'official.vision.image_classification.efficientnet.efficientnet_model',
    'official.modeling.optimization.ema_optimizer',
]:
    _stub_module(_n)

# 'lvis' -- LVIS evaluator. Subclassed at module scope, needs real classes.
_stub_package('lvis')
_stub_module('lvis.lvis')
_stub_module('lvis.eval')
_stub_module('lvis.results')


class _LVISStub:
    def __init__(self, *args, **kwargs):
        raise RuntimeError("lvis is stubbed out; not used for SSD export.")


sys.modules['lvis'].LVIS = _LVISStub
sys.modules['lvis'].LVISEval = _LVISStub
sys.modules['lvis'].LVISResults = _LVISStub
sys.modules['lvis.lvis'].LVIS = _LVISStub
sys.modules['lvis.eval'].LVISEval = _LVISStub
sys.modules['lvis.results'].LVISResults = _LVISStub


# ---------------------------------------------------------------------------
# Shim B: tf.estimator / tf.compat.v1.estimator (removed in TF 2.16+).
# ---------------------------------------------------------------------------
import tensorflow as tf


class _ModeKeys:
    TRAIN = 'train'
    EVAL = 'eval'
    PREDICT = 'infer'


for _mod_name, _attr_path in [
    ('tensorflow.compat.v1.estimator', ('compat', 'v1', 'estimator')),
    ('tensorflow.estimator', ('estimator',)),
]:
    _stub = types.ModuleType(_mod_name)
    _stub.ModeKeys = _ModeKeys
    sys.modules[_mod_name] = _stub
    _obj = tf
    for _p in _attr_path[:-1]:
        _obj = getattr(_obj, _p)
    setattr(_obj, _attr_path[-1], _stub)


# ---------------------------------------------------------------------------
# Shim C: control_flow_ops legacy attribute aliases (tf_slim still uses them).
# ---------------------------------------------------------------------------
from tensorflow.python.ops import control_flow_ops as _cfo

for _name in ['case', 'cond', 'group', 'no_op', 'tuple', 'while_loop']:
    if not hasattr(_cfo, _name):
        _public = getattr(tf, _name, None)
        if _public is not None:
            setattr(_cfo, _name, _public)


# ---------------------------------------------------------------------------
# Shim D: Keras 3 removed tf.keras.backend.set_learning_phase. Under
# TF_USE_LEGACY_KERAS this is a no-op (Keras 2 has it), but we keep the guard
# in case the env var is ever flipped off.
# ---------------------------------------------------------------------------
if not hasattr(tf.keras.backend, 'set_learning_phase'):
    tf.keras.backend.set_learning_phase = lambda *args, **kwargs: None
if not hasattr(tf.keras.backend, 'learning_phase'):
    tf.keras.backend.learning_phase = lambda: 0


# ---------------------------------------------------------------------------
# Delegate to exporter_main_v2.
# ---------------------------------------------------------------------------
from absl import app
from object_detection import exporter_main_v2  # noqa: E402

if __name__ == '__main__':
    app.run(exporter_main_v2.main)
'''

with open(EXPORTER_WRAPPER_PATH, "w") as f:
    f.write(_exporter_wrapper_src)

print(f"Wrote exporter wrapper: {EXPORTER_WRAPPER_PATH} ({len(_exporter_wrapper_src)} chars)")

Wrote exporter wrapper: /kaggle/working/export_with_shims.py (4606 chars)


In [28]:
# Cell #11.6: Write a wrapper for export_tflite_graph_tf2.py.
# This is a separate exporter from exporter_main_v2.py — it produces a
# SavedModel with the TFLite_Detection_PostProcess custom op and static
# [1, 640, 640, 3] input, both required for EdgeTPU compilation.

TFLITE_EXPORTER_WRAPPER_PATH = "/kaggle/working/export_tflite_with_shims.py"

_tflite_exporter_wrapper_src = r'''"""
Launcher for object_detection.export_tflite_graph_tf2 with TF 2.19 / Keras 3
compat shims. Identical shim layer to export_with_shims.py; only the target
main differs.
"""
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import sys
import types


def _stub_package(name):
    if name not in sys.modules:
        pkg = types.ModuleType(name)
        pkg.__path__ = []
        sys.modules[name] = pkg
    if '.' in name:
        parent, leaf = name.rsplit('.', 1)
        setattr(sys.modules[parent], leaf, sys.modules[name])


def _stub_module(name):
    if name not in sys.modules:
        sys.modules[name] = types.ModuleType(name)
    parent, leaf = name.rsplit('.', 1)
    setattr(sys.modules[parent], leaf, sys.modules[name])


for _n in [
    'official', 'official.legacy', 'official.legacy.image_classification',
    'official.legacy.image_classification.efficientnet',
    'official.vision', 'official.vision.image_classification',
    'official.vision.image_classification.efficientnet',
    'official.modeling', 'official.modeling.optimization',
]:
    _stub_package(_n)
for _n in [
    'official.legacy.image_classification.efficientnet.efficientnet_model',
    'official.vision.image_classification.efficientnet.efficientnet_model',
    'official.modeling.optimization.ema_optimizer',
]:
    _stub_module(_n)

_stub_package('lvis')
_stub_module('lvis.lvis')
_stub_module('lvis.eval')
_stub_module('lvis.results')


class _LVISStub:
    def __init__(self, *args, **kwargs):
        raise RuntimeError("lvis is stubbed out; not used for SSD export.")


sys.modules['lvis'].LVIS = _LVISStub
sys.modules['lvis'].LVISEval = _LVISStub
sys.modules['lvis'].LVISResults = _LVISStub
sys.modules['lvis.lvis'].LVIS = _LVISStub
sys.modules['lvis.eval'].LVISEval = _LVISStub
sys.modules['lvis.results'].LVISResults = _LVISStub


import tensorflow as tf


class _ModeKeys:
    TRAIN = 'train'
    EVAL = 'eval'
    PREDICT = 'infer'


for _mod_name, _attr_path in [
    ('tensorflow.compat.v1.estimator', ('compat', 'v1', 'estimator')),
    ('tensorflow.estimator', ('estimator',)),
]:
    _stub = types.ModuleType(_mod_name)
    _stub.ModeKeys = _ModeKeys
    sys.modules[_mod_name] = _stub
    _obj = tf
    for _p in _attr_path[:-1]:
        _obj = getattr(_obj, _p)
    setattr(_obj, _attr_path[-1], _stub)


from tensorflow.python.ops import control_flow_ops as _cfo

for _name in ['case', 'cond', 'group', 'no_op', 'tuple', 'while_loop']:
    if not hasattr(_cfo, _name):
        _public = getattr(tf, _name, None)
        if _public is not None:
            setattr(_cfo, _name, _public)


if not hasattr(tf.keras.backend, 'set_learning_phase'):
    tf.keras.backend.set_learning_phase = lambda *args, **kwargs: None
if not hasattr(tf.keras.backend, 'learning_phase'):
    tf.keras.backend.learning_phase = lambda: 0


from absl import app
from object_detection import export_tflite_graph_tf2  # noqa: E402

if __name__ == '__main__':
    app.run(export_tflite_graph_tf2.main)
'''

with open(TFLITE_EXPORTER_WRAPPER_PATH, "w") as f:
    f.write(_tflite_exporter_wrapper_src)

print(f"Wrote TFLite exporter wrapper: {TFLITE_EXPORTER_WRAPPER_PATH} "
      f"({len(_tflite_exporter_wrapper_src)} chars)")


# Run the TFLite-targeting export. Output goes to a SEPARATE directory so it
# doesn't overwrite the inference SavedModel from cell #12 (you may want to
# keep both — one for general TF inference, one for TFLite/EdgeTPU).
TFLITE_EXPORT_DIR = "/kaggle/working/exported_model_tflite"
os.makedirs(TFLITE_EXPORT_DIR, exist_ok=True)

print(f"\nExporting TFLite-targeting SavedModel to {TFLITE_EXPORT_DIR}...")
print("=" * 60)

!python "{TFLITE_EXPORTER_WRAPPER_PATH}" \
    --pipeline_config_path="{PIPELINE_CONFIG_PATH}" \
    --trained_checkpoint_dir="{TRAIN_DIR}" \
    --output_directory="{TFLITE_EXPORT_DIR}" \
    --max_detections=20 \
    --ssd_use_regular_nms=False

print(f"\nTFLite-targeting SavedModel exported.")
!ls -la "{TFLITE_EXPORT_DIR}/saved_model/"

Wrote TFLite exporter wrapper: /kaggle/working/export_tflite_with_shims.py (2985 chars)

Exporting TFLite-targeting SavedModel to /kaggle/working/exported_model_tflite...
2026-04-25 20:00:17.079950: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777147217.102660   90644 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777147217.109720   90644 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777147217.127530   90644 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777147217.127558   90644 computation_placer.cc:177] computation placer already

In [27]:
# Step 1: Export SavedModel via the compat wrapper from cell #11.5.
print("Exporting SavedModel...")
os.makedirs(EXPORT_DIR, exist_ok=True)

!python "{EXPORTER_WRAPPER_PATH}" \
    --pipeline_config_path="{PIPELINE_CONFIG_PATH}" \
    --trained_checkpoint_dir="{TRAIN_DIR}" \
    --output_directory="{EXPORT_DIR}" \
    --input_type=image_tensor

print(f"\nSavedModel exported to: {EXPORT_DIR}")
!ls -la "{EXPORT_DIR}/saved_model/"

Exporting SavedModel...
2026-04-25 19:44:37.369187: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777146277.390546   90602 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777146277.397363   90602 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777146277.414172   90602 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777146277.414228   90602 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777146277.414236   90602 computation_placer.cc:177

In [29]:
# Step 2: Convert to TFLite INT8 using the TFLite-targeting export from #11.6.
# Full integer quantization needs a representative dataset to calibrate
# activation ranges; we sample 100 images from the training TFRecords.

TFLITE_DIR = os.path.join(WORK_DIR, "tflite_model")
os.makedirs(TFLITE_DIR, exist_ok=True)

# IMPORTANT: convert from the TFLite-targeting export, NOT the standard one.
# The standard export from cell #12 has variable input shape and uses NMS ops
# the EdgeTPU compiler can't handle.
SAVED_MODEL_DIR = os.path.join(TFLITE_EXPORT_DIR, "saved_model")

print("Converting to TFLite with INT8 quantization...")


def representative_dataset_gen():
    """Yield 100 calibration samples for INT8 activation range estimation.
    Uses parse_single_example with the same feature spec OD's input pipeline
    uses, then resizes to 640x640 and casts to float32. Quantization happens
    automatically based on the inference_input_type setting below."""
    feature_description = {
        "image/encoded": tf.io.FixedLenFeature([], tf.string),
    }
    # Use a few shards interleaved so we sample across the dataset, not just
    # the first shard's contents (which may be biased to a single source).
    dataset = tf.data.TFRecordDataset(train_records[:4])
    dataset = dataset.shuffle(buffer_size=500, seed=42)

    count = 0
    for raw_record in dataset:
        if count >= 100:
            break
        parsed = tf.io.parse_single_example(raw_record, feature_description)
        image = tf.image.decode_jpeg(parsed["image/encoded"], channels=3)
        image = tf.image.resize(image, [640, 640])
        # The TFLite-targeting export expects float32 [0, 255] input (it bakes
        # the /127.5-1 normalization into the graph). Quantization to uint8
        # happens via inference_input_type below.
        image = tf.cast(image, tf.float32)
        image = tf.expand_dims(image, axis=0)
        yield [image]
        count += 1
    print(f"  Calibration: yielded {count} representative samples.")


converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
# Allow the built-in TFLite ops AND the custom NMS op the OD TFLite exporter
# inserts. Without TFLITE_BUILTINS, the NMS won't be representable.
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS,
]
converter.inference_input_type = tf.uint8     # EdgeTPU expects uint8 input
converter.inference_output_type = tf.float32  # detections come out as float

try:
    tflite_model = converter.convert()
    TFLITE_MODEL_PATH = os.path.join(
        TFLITE_DIR, "ssd_mobilenet_v2_fpnlite_640_person_int8.tflite")
    with open(TFLITE_MODEL_PATH, "wb") as f:
        f.write(tflite_model)
    model_size_mb = len(tflite_model) / (1024 * 1024)
    print(f"\nTFLite INT8 model saved: {TFLITE_MODEL_PATH}")
    print(f"Model size: {model_size_mb:.1f} MB")
except Exception as e:
    print(f"\nTFLite conversion failed: {e}")
    print("Most common causes: unsupported op (try adding "
          "tf.lite.OpsSet.SELECT_TF_OPS to supported_ops; reduces EdgeTPU "
          "coverage but still works on CPU), or representative dataset "
          "yielded zero samples.")
    TFLITE_MODEL_PATH = None

Converting to TFLite with INT8 quantization...


W0000 00:00:1777147281.691423      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1777147281.691474      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1777147282.011844      55 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


  Calibration: yielded 100 representative samples.

TFLite INT8 model saved: /kaggle/working/tflite_model/ssd_mobilenet_v2_fpnlite_640_person_int8.tflite
Model size: 5.4 MB


fully_quantize: 0, inference_type: 6, input_inference_type: UINT8, output_inference_type: FLOAT32


In [31]:
# Step 3: Install EdgeTPU compiler if needed, then compile the INT8 TFLite.
# The compiler is an x86_64 binary distributed by Google via apt. Kaggle's
# image is x86_64 Linux but doesn't have the Coral apt repo configured by
# default. We add it, install the compiler, then run it.

import shutil
import subprocess


def _have_edgetpu_compiler():
    return shutil.which("edgetpu_compiler") is not None


if not TFLITE_MODEL_PATH or not os.path.exists(TFLITE_MODEL_PATH):
    print("No TFLite model to compile. Fix the conversion step above first.")
else:
    if not _have_edgetpu_compiler():
        print("EdgeTPU compiler not found. Installing...")
        # Add Google's Coral apt repo and install the compiler. The
        # signed-by= form is the modern apt-key replacement.
        !curl -sS https://packages.cloud.google.com/apt/doc/apt-key.gpg \
            | sudo gpg --dearmor -o /usr/share/keyrings/coral-edgetpu-archive-keyring.gpg 2>&1 | tail -3
        !echo "deb [signed-by=/usr/share/keyrings/coral-edgetpu-archive-keyring.gpg] \
https://packages.cloud.google.com/apt coral-edgetpu-stable main" \
            | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list > /dev/null
        !sudo apt-get update -qq 2>&1 | tail -3
        !sudo apt-get install -y -qq edgetpu-compiler 2>&1 | tail -3

    # Verify install.
    !edgetpu_compiler --version

    # Compile. -s prints op-level details; -o sets output dir. The compiler
    # writes <name>_edgetpu.tflite alongside a <name>_edgetpu.log file.
    print(f"\nCompiling {os.path.basename(TFLITE_MODEL_PATH)} for EdgeTPU...")
    print("=" * 60)
    !edgetpu_compiler -s -o "{TFLITE_DIR}" "{TFLITE_MODEL_PATH}"
    print("=" * 60)

    EDGETPU_MODEL_PATH = TFLITE_MODEL_PATH.replace(".tflite", "_edgetpu.tflite")
    EDGETPU_LOG_PATH = TFLITE_MODEL_PATH.replace(".tflite", "_edgetpu.log")

    if os.path.exists(EDGETPU_MODEL_PATH):
        size_mb = os.path.getsize(EDGETPU_MODEL_PATH) / (1024 * 1024)
        print(f"\n✓ EdgeTPU model: {EDGETPU_MODEL_PATH}")
        print(f"  Size: {size_mb:.1f} MB")
        if os.path.exists(EDGETPU_LOG_PATH):
            print(f"\nCompiler log:")
            print(f"  {EDGETPU_LOG_PATH}")
    else:
        print("\n✗ EdgeTPU compilation did not produce an output file.")
        print("  Check the compiler output above for the failing op or error.")
        EDGETPU_MODEL_PATH = None

EdgeTPU compiler not found. Installing...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

Edge TPU Compiler version 16.0.384591198

Compiling ssd_mobilenet_v2_fpnlite_640_person_int8.tflite for EdgeTPU...
Edge TPU Compiler version 16.0.384591198
Started a compilation timeout timer of 180 seconds.

Model compiled successfully in 2637 ms.

Input model: /kaggle/working/tflite_model/ssd_mobilenet_v2_fpnlite_640_person_int8.tflite
Input size: 5.40MiB
Output model: /kaggle/working/tflite_model/ssd_mobilenet_v2_fpnlite_640_person_int8_edgetpu.tflite
Output size: 10.19MiB
On-chip memory used for caching model parameters: 6.69MiB
On-chip memory remaining for caching model parameters: 182.25KiB
Off-chip memory used for streaming uncached model parameters: 0.00B
Number of Edge TPU su

## 10. Summary & Output Files

### Files to download for deployment on Raspberry Pi


In [32]:
print("=" * 60)
print("TRAINING COMPLETE — Output Summary")
print("=" * 60)

# Some variables may not exist if earlier cells were skipped or failed.
_edgetpu = locals().get("EDGETPU_MODEL_PATH") or "Not created"

outputs = {
    "Pipeline config":  PIPELINE_CONFIG_PATH,
    "Training dir":     TRAIN_DIR,
    "SavedModel (TF)":  os.path.join(EXPORT_DIR, "saved_model"),
    "SavedModel (TFLite-targeting)": os.path.join(TFLITE_EXPORT_DIR, "saved_model"),
    "TFLite INT8":      TFLITE_MODEL_PATH if TFLITE_MODEL_PATH else "Not created",
    "TFLite EdgeTPU":   _edgetpu,
    "Label map":        LABEL_MAP_PATH,
}

for name, path in outputs.items():
    if isinstance(path, str) and os.path.exists(path):
        if os.path.isdir(path):
            size = sum(
                os.path.getsize(os.path.join(dp, f))
                for dp, _, fns in os.walk(path)
                for f in fns
            )
        else:
            size = os.path.getsize(path)
        print(f"  {name}: {path} ({size/(1024*1024):.1f} MB)")
    else:
        print(f"  {name}: {path}")

print()
print("Next steps:")
print("  1. Save this notebook version (Save Version → Quick Save with output)")
print("  2. Download the TFLite INT8 + EdgeTPU models for the Pi")
print("  3. Run Notebook 3 (EfficientDet-Lite0) for model comparison")
print("  4. Run Notebook 4 for bucketed evaluation by pixel-height")

print(f"\nDisk usage:")
!du -sh /kaggle/working/*/

TRAINING COMPLETE — Output Summary
  Pipeline config: /kaggle/working/pipeline.config (0.0 MB)
  Training dir: /kaggle/working/training_output (421.1 MB)
  SavedModel (TF): /kaggle/working/exported_model/saved_model (20.5 MB)
  SavedModel (TFLite-targeting): /kaggle/working/exported_model_tflite/saved_model (21.7 MB)
  TFLite INT8: /kaggle/working/tflite_model/ssd_mobilenet_v2_fpnlite_640_person_int8.tflite (5.4 MB)
  TFLite EdgeTPU: /kaggle/working/tflite_model/ssd_mobilenet_v2_fpnlite_640_person_int8_edgetpu.tflite (10.2 MB)
  Label map: /kaggle/input/notebooks/giladfaibish/data-preparation/snipeit_person_dataset/label_map.pbtxt (0.0 MB)

Next steps:
  1. Save this notebook version (Save Version → Quick Save with output)
  2. Download the TFLite INT8 + EdgeTPU models for the Pi
  3. Run Notebook 3 (EfficientDet-Lite0) for model comparison
  4. Run Notebook 4 for bucketed evaluation by pixel-height

Disk usage:
35M	/kaggle/working/exported_model/
22M	/kaggle/working/exported_model_tfl